In [ ]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON
from urllib.error import HTTPError
import time
import datetime

INPUT_FILE = 'wikidata8.csv'
OUTPUT_FILE = 'wikidata9.csv'
BATCH_SIZE = 300

COLS_TO_MAP = [
    'genero_P21', 'nacionalidade_P27', 'sobrenome_P734', 
    'nome_P735', 'local_nascimento_P19', 'ocupacao_P106', 
    'residencia_P551'
]

def get_sparql_data(qid_batch):
    """
    Recebe uma lista de QIDs e retorna um dicionário {QID: Label}.
    Usa a lógica de retries e tratamento de erro solicitada.
    """
    endpoint_url = "https://query.wikidata.org/sparql"
    values_str = " ".join([f"wd:{qid}" for qid in qid_batch])
    
    query = f"""
    SELECT ?item ?itemLabel WHERE {{
      VALUES ?item {{ {values_str} }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,pt,es,fr,de,it,ru,ja,zh". }}
    }}
    """

    sparql = SPARQLWrapper(endpoint_url)
    sparql.setQuery(query)
    sparql.setMethod('GET')
    sparql.setReturnFormat(JSON)

    max_retries = 5
    for attempt in range(max_retries):
        try:
            results = sparql.query().convert()
            mapping = {}
            for result in results["results"]["bindings"]:
                q_id = result["item"]["value"].split("/")[-1]
                label = result["itemLabel"]["value"]
                mapping[q_id] = label
            return mapping

        except HTTPError as e:
            if e.code == 429: 
                print(f"  -> ERRO 429 (Too Many Requests). Tentativa {attempt + 1}/{max_retries}. Aguardando...")
                time.sleep(2 * (attempt + 1))
            else:
                print(f"  -> ERRO HTTP Inesperado: {e}")
                break
        except Exception as e:
            print(f"  -> ERRO (não-HTTP): {e}")
            break 
            
        time.sleep(0.5)
    
    return {}

print("1. Carregando CSV...")
df = pd.read_csv(INPUT_FILE)

print("2. Mapeando QIDs únicos...")
unique_qids = set()

for col in COLS_TO_MAP:
    if col in df.columns:
        items = df[col].dropna().astype(str).str.split(';').explode().unique()
        for item in items:
            if isinstance(item, str) and item.startswith('Q') and item[1:].isdigit():
                unique_qids.add(item)

qid_list = list(unique_qids)
total_qids = len(qid_list)
print(f"-> Total de QIDs únicos para buscar: {total_qids}")

def format_time(seconds):
    return str(datetime.timedelta(seconds=int(seconds)))

qid_to_label = {}
start_time = time.time()
total_found = 0

print(f"3. Iniciando busca na Wikidata para {total_qids} itens...")
print("-" * 80)

for i in range(0, total_qids, BATCH_SIZE):
    batch = qid_list[i:i + BATCH_SIZE]
    
    batch_results = get_sparql_data(batch)
    qid_to_label.update(batch_results)
    
    current_count = i + len(batch)
    if current_count > total_qids: current_count = total_qids
    
    elapsed_time = time.time() - start_time
    if elapsed_time > 0:
        items_per_second = current_count / elapsed_time
        remaining_items = total_qids - current_count
        eta_seconds = remaining_items / items_per_second
    else:
        items_per_second = 0
        eta_seconds = 0
        
    percent = (current_count / total_qids) * 100
    total_found = len(qid_to_label)
    
    print(f"[{percent:5.1f}%] {current_count}/{total_qids} | "
          f"Achados: {total_found} | "
          f"Vel: {items_per_second:.1f} id/s | "
          f"Tempo: {format_time(elapsed_time)} | "
          f"ETA: {format_time(eta_seconds)}")

print("-" * 80)
print(f"-> Busca concluída. {len(qid_to_label)} labels recuperados em {format_time(time.time() - start_time)}.")

print(f"-> Busca concluída. {len(qid_to_label)} labels recuperados.")

print("4. Aplicando substituições no Dataset...")

def replace_ids(cell_value):
    if pd.isna(cell_value):
        return cell_value
    
    original_values = str(cell_value).split(';')
    new_values = []
    
    for val in original_values:
        val = val.strip()
        new_values.append(qid_to_label.get(val, val))
        
    return ";".join(new_values)

for col in COLS_TO_MAP:
    if col in df.columns:
        df[col] = df[col].apply(replace_ids)

print(f"5. Salvando em {OUTPUT_FILE}...")
df.to_csv(OUTPUT_FILE, index=False)
print("Concluído com sucesso!")

1. Carregando CSV...


C:\Users\Leo\AppData\Local\Temp\ipykernel_12372\2361442918.py:64: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_FILE)


2. Mapeando QIDs únicos...
-> Total de QIDs únicos para buscar: 891030
3. Iniciando busca na Wikidata para 891030 itens...
--------------------------------------------------------------------------------
[  0.0%] 300/891030 | Achados: 300 | Vel: 138.5 id/s | Tempo: 0:00:02 | ETA: 1:47:09
[  0.1%] 600/891030 | Achados: 600 | Vel: 143.5 id/s | Tempo: 0:00:04 | ETA: 1:43:26
[  0.1%] 900/891030 | Achados: 900 | Vel: 129.7 id/s | Tempo: 0:00:06 | ETA: 1:54:21
[  0.1%] 1200/891030 | Achados: 1200 | Vel: 140.3 id/s | Tempo: 0:00:08 | ETA: 1:45:41
[  0.2%] 1500/891030 | Achados: 1500 | Vel: 149.5 id/s | Tempo: 0:00:10 | ETA: 1:39:09
[  0.2%] 1800/891030 | Achados: 1800 | Vel: 153.0 id/s | Tempo: 0:00:11 | ETA: 1:36:52
[  0.2%] 2100/891030 | Achados: 2100 | Vel: 145.6 id/s | Tempo: 0:00:14 | ETA: 1:41:46
[  0.3%] 2400/891030 | Achados: 2400 | Vel: 148.6 id/s | Tempo: 0:00:16 | ETA: 1:39:40
[  0.3%] 2700/891030 | Achados: 2700 | Vel: 151.4 id/s | Tempo: 0:00:17 | ETA: 1:37:47
[  0.3%] 3000/89103